# 08 pamoka – daugiaveiksmio agento dizaino šablonas


## Sąranka


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Kodėl daugiaagentinės sistemos?

Tikro gyvenimo užduotys, tokios kaip kelionės planavimas, apima daug skirtingų sričių — logistiką, vietos žinias, biudžetavimą ir kt. Vienas agentas, stengdamasis viską apimti, greitai tampa nevaldomas.

Daugiaagentinės sistemos tai sprendžia per **specializaciją**: kiekvienas agentas susitelkia į vieną sritį, taip gamindamas aukštesnės kokybės rezultatus nei generalistas. Jos taip pat gerina **masto galimybes** — galite pridėti naujų agentų (pvz., skrydžių specialistą, restoranų kritiką) nepridarydami esamos darbo eigos. Agentai sudaro struktūruotą grandinę, perduodami kontekstą vienas kitam.


## Specializuotų agentų kūrimas


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Sekvenčio Darbo Eigos Kūrimas

`WorkflowBuilder` leidžia sujungti agentus į nukreiptą grafiką. Čia kuriame paprastą dviejų žingsnių grandinę: **TravelPlanner** sudaro kelionės maršrutą, tada **TravelConcierge** jį peržiūri ir patobulina.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Daugiau agentų pridėjimas prie darbo eigos

Viena didžiausių daugiaagentės struktūros privalumų yra jos paprastumas išplėsti. Žemiau pridedame **BudgetReviewer** agentą, kuris tikrina planą pagal keliautojo biudžetą, pažymi punktus, galinčius viršyti išlaidų ribą, ir siūlo pinigų taupymo alternatyvas. Dabar darbo eiga veikia trijų agentų seka:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Santrauka

Šioje pamokoje sužinojote, kaip:

1. **Kurti specializuotus agentus** — kiekvienas turintis aiškų vaidmenį (planavimas, konsjeržas, biudžeto peržiūra).
2. **Jungti agentus į seką** naudojant `WorkflowBuilder` ir `add_edge`.
3. **Srautiniu būdu perduoti išvestį** iš kelių agentų vamzdyno, sekant, kuris agentas kalba.
4. **Išplėsti darbo procesą** pridėjus naujų agentų grandinei be esamų keitimo.

Kelių agentų dizaino šablonas palaiko kiekvieną agentą paprastą, tačiau sukuria turtingesnius, kruopščiau peržiūrėtus rezultatus nei bet kuris vienas agentas galėtų pasiekti vienas.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors stengiamės užtikrinti tikslumą, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų arba netikslumų. Originalus dokumentas jo gimtąja kalba turi būti laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama pasitelkti profesionalų žmogaus vertimą. Mes neatsakome už jokią painiavą ar neteisingą interpretaciją, kilusią naudojant šį vertimą.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
